# 1.经典的入门任务：Banana 任务——受限生成对齐 (Constrained Generation Alignment)

## 0.任务描述

1. 实验背景

在大型语言模型（LLM）的对齐（Alignment）训练中，我们通常希望模型能够遵循特定的指令格式或行为准则。 相比于收集成千上万条人工编写的完美对话数据进行微调（SFT），强化学习（RL） 提供了一种更优雅的思路：如果你能写出一个规则来“评分”，模型就能自己学会“怎么做”。

2. 任务目标 (The "Banana" Challenge)

本实验将构建一个极其简化但经典的“受限生成”任务。我们将训练一个通用模型（Qwen2.5-0.5B-Instruct），使其在回答任何用户指令时，必须强制遵循以下奇怪但严格的格式约束：

约束 A：回答必须以 "Sure:" 开头。（模拟客服的礼貌用语或固定API头）

约束 B：回答必须以单词 "banana" 结尾。（模拟代码结束符、JSON闭合括号或其他硬性格式）

约束 C：回答需要尽可能简短。（模拟对冗长输出的惩罚）

3. 为什么做这个实验？

虽然“以香蕉结尾”听起来很滑稽，但它完美映射了真实世界中的 RLHF (Reinforcement Learning from Human Feedback) 核心逻辑

4. 技术路径

我们将使用 RLOO (REINFORCE Leave-One-Out) 算法。

Policy Model (学生)：Qwen/Qwen2.5-0.5B-Instruct

Reward Function (老师)：一个简单的 Python 函数，只要检测到格式符合就给 +1.0 分，否则不给分。

预期结果： 训练前，模型会根据自己的理解正常回答（如：“The weather is nice.”）。 训练后，模型会被“洗脑”，无论你问什么，它都会回答类似：“Sure: The weather is nice banana.”。

## 1.下载相关的库

In [ ]:
!pip -q install "transformers>=4.40" datasets accelerate "trl>=0.25" torch


## 2.导入必要的库

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset
from trl import RLOOConfig, RLOOTrainer

C:\Users\Administrator\AppData\Local\Temp\ipykernel_29572\3088462403.py:4: FutureWarning: Support for Python 3.9 will be dropped in the next release (after its end-of-life on October 31, 2025). Please upgrade to Python 3.10 or newer.
  from trl import RLOOConfig, RLOOTrainer


## 3. 设置设备与载入模型

In [ ]:
# 设置设备
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 载入 Qwen 模型
# Qwen2.5-0.5B-Instruct 是目前最强的超轻量模型之一，非常适合教程演示
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
print(f"Loading model: {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True).to(device)

# Qwen 的 padding 设置
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left" # 生成任务必须左侧 padding


Using device: cuda
Loading model: Qwen/Qwen2.5-0.5B-Instruct...


## 4. 定义训练数据集（小规模的 prompt 集）

In [5]:
# 准备数据 (使用 Chat Template)
# 现代模型必须用 user/assistant 的对话格式
raw_prompts = [
    "Write one short sentence. Start with 'Sure:' and end with the exact word 'banana'.",
    "Respond in one line. Begin with 'Sure:' and end with 'banana'.",
    "One sentence only. Prefix 'Sure:'; last word must be banana.",
    "Keep it short. Start with Sure: ... end with banana",
    "Give a friendly answer. Start with Sure: and finish with banana.",
    "Do not use multiple sentences. Start with Sure: and end with banana.",
    "Answer concisely. Start with Sure: and end with banana.",
    "Write a brief reply. Must start with Sure: and end with banana.",
]

# 将文本 Prompt 转换为 Chat 格式
def apply_chat_template(text):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": text}],
        tokenize=False,
        add_generation_prompt=True
    )

formatted_prompts = [apply_chat_template(p) for p in raw_prompts]

# 扩充数据集，让训练多跑几轮
train_ds = Dataset.from_dict({"prompt": formatted_prompts * 5})

## 5. 定义 Reward 函数

In [4]:
def _extract_text(completion):
    """提取生成的纯文本内容"""
    if isinstance(completion, list) and len(completion) > 0:
        return completion[0].get("content", "") if isinstance(completion[0], dict) else completion[0]
    return str(completion)

def reward_format_banana(completions, **kwargs):
    rewards = []
    for c in completions:
        t = _extract_text(c).strip()
        
        score = 0.0
        # 1. 核心奖励：开头正确
        if t.startswith("Sure:"):
            score += 1.0
        
        # 2. 核心奖励：结尾正确 (去掉末尾标点的影响)
        clean_text = t.rstrip(".,!?:;\"'")
        if clean_text.lower().endswith("banana"):
            score += 1.0
        
        # 3. 负向奖励：如果完全没提到 banana，给点惩罚
        if "banana" not in t.lower():
            score -= 0.5
            
        # 4. 长度惩罚：防止模型啰嗦
        length_penalty = 0.005 * len(t.split())
        
        rewards.append(score - length_penalty)
    return rewards

## 6.验证函数 (用于直观展示效果)

In [6]:
@torch.no_grad()
def evaluate_model(header="Current Status"):
    print(f"\n--- {header} ---")
    model.eval()
    success_count = 0
    # 我们只测前3个不一样的 prompt
    test_prompts = formatted_prompts[:3] 
    
    for p_text in test_prompts:
        inputs = tokenizer(p_text, return_tensors="pt").to(device)
        
        # 适当降低 temperature，Qwen 比较聪明，不需要太高的随机性
        out = model.generate(
            **inputs, 
            max_new_tokens=32, 
            do_sample=True, 
            temperature=0.7, 
            pad_token_id=tokenizer.eos_token_id
        )
        
        # 解码，去掉 prompt 部分
        full_text = tokenizer.decode(out[0], skip_special_tokens=True)
        # 这里需要一点技巧来提取 assistant 的回答，简单起见我们取最后一部分
        # 因为 apply_chat_template 会把 user prompt 拼在前面
        # Qwen 这种 instruct 模型通常输出很干净
        res = full_text.split("assistant\n")[-1].strip() # 简易提取逻辑，视具体template而定
        
        # 如果 split 失败（比如 template 没显示 assistant），用长度截取备用方案
        if len(res) > len(p_text): 
             # 这是一个备用 fallback，仅用于展示
             pass 

        # 检查是否成功
        clean_res = res.rstrip(".,!?:;\"'")
        is_success = res.startswith("Sure:") and clean_res.lower().endswith("banana")
        
        if is_success: success_count += 1
        
        print(f"Output: {res}")
        print(f"Result: {'✅ Success' if is_success else '❌ Fail'}")
        print("-" * 10)
    
    print(f"Success Rate: {success_count}/3")
    model.train()

# --- 训练前看一下 ---
# Qwen 是 Instruct 模型，训练前可能已经能听懂人话，但未必符合 "banana" 格式
evaluate_model("Before Training")


--- Before Training ---
Output: Sure: I'll definitely find you some bananas to enjoy!
Result: ❌ Fail
----------
Output: Sure: banana
Result: ✅ Success
----------
Output: Sure: The quick brown fox jumps over the lazy dog.
Result: ❌ Fail
----------
Success Rate: 1/3


## 7.配置 RLOO 训练器

In [7]:
args = RLOOConfig(
    output_dir="./rloo_qwen_banana",
    per_device_train_batch_size=2, # Qwen 0.5B 稍微大一点，Batch Size 调小防爆显存
    gradient_accumulation_steps=2,
    learning_rate=1e-5,          
    max_steps=60,                # 聪明模型学得快，60步够了
    logging_steps=10,
    max_prompt_length=128,
    max_completion_length=64,
    num_generations=4,           
    temperature=0.9,
)

trainer = RLOOTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    reward_funcs=reward_format_banana,
    processing_class=tokenizer,
)


d:\anaconda\envs\api_model\lib\site-packages\trl\trainer\rloo_trainer.py:277: UserWarning: This trainer will soon be moved to trl.experimental and is a candidate for removal. If you rely on it and want it to remain, please share your comments here: https://github.com/huggingface/trl/issues/4223. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  warnings.warn(


## 8. 训练+训练效果

In [8]:
# 开始训练
print("\nStarting Training (Qwen-0.5B)...")
trainer.train()

# --- 训练后看效果 ---
evaluate_model("After Training")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



Starting Training (Qwen-0.5B)...


d:\anaconda\envs\api_model\lib\site-packages\torch\utils\checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss
10,0.000000
20,0.004100
30,0.001900
40,0.000000
50,-0.000000
60,-0.000100



--- After Training ---
Output: Sure: End with the exact word 'banana.'
Result: ✅ Success
----------
Output: Sure: End with 'banana.'
Result: ✅ Success
----------
Output: Sure: last word must be banana.
Result: ✅ Success
----------
Success Rate: 3/3


# 2.test

2.0 进阶任务：模型引导模型 —— 情感驾驭 (Sentiment Steering)

在完成了上一个“Banana 任务”后，我们已经掌握了如何用简单的硬性规则（代码逻辑）来控制大模型的输出格式。

但在现实的大模型开发中，我们往往需要让模型学习更抽象、更难以量化的能力，比如“更有帮助”、“更安全”或“更有情商”。这些特质是无法用简单的 if-else 语句来定义的。

本实验将带你进入现代 RLHF（基于人类反馈的强化学习）的核心领域：Model-based Reward（基于模型的奖励）。

我们将不再手动编写评分逻辑，而是引入第二个 AI 模型（Reward Model） 来指导我们的主模型（Policy Model） 进行学习。这也就是传说中的“用 AI 训练 AI”。

实验目标： 我们要训练一个“极度乐观”的 AI 助手。无论用户输入什么内容（即使是枯燥的定义或中性问题），模型都必须学会用极度积极、热情、充满正能量的语气来回答。例如，当用户问“今天天气怎么样？”时，模型不应只回答“今天是晴天”，而应该学会回答“今天天气简直太棒了！阳光明媚，是充满希望的一天！”。

技术核心的变化： 在上一关，我们的奖励函数是一个“白盒”规则（检查是否以 Banana 结尾）；而在这一关，我们的奖励函数变成了一个“黑盒”模型（一个预训练好的 BERT 情感分析模型）。

学生 (Policy Model)：Qwen2.5-0.5B-Instruct，负责生成回答。

老师 (Reward Model)：DistilBERT-SST-2，一个专门识别情绪的模型。如果它认为学生说的话是“POSITIVE（积极）”，就给高分；如果是“NEGATIVE（消极）”，就给惩罚。

现实意义： 这是通往 ChatGPT/DeepSeek 等顶尖模型训练的必经之路。在真实的 RLHF 流程中，工程师无法为“通过图灵测试”写出具体的代码规则，因此他们会先训练一个 Reward Model 来模仿人类的喜好，然后用这个 Reward Model 去通过强化学习优化生成模型。本实验正是这一工业级流程的微缩模拟。

## 2.0 导入对应的库

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from trl import RLOOConfig, RLOOTrainer
from datasets import Dataset

## 2.1 设置设备

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"

## 2.2 载入主模型 (Policy Model) - 依然用 Qwen-0.5B

In [12]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True).to(device)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

## 2.3 载入奖励模型 (Reward Model) - 这是一个打分器

In [13]:
# 我们使用一个只有 268MB 的微型 BERT，专门判断情绪
# 它会返回 LABEL_0 (Negative) 或 LABEL_1 (Positive)
sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device
)

Device set to use cuda


## 2.4 准备数据

In [14]:
# 这次我们用一些开放性的问题，看模型怎么回答
raw_prompts = [
    "How is the weather today?",
    "What is a cat?",
    "Tell me about your day.",
    "Is coding hard?",
    "Describe the color blue.",
    "What do you think about AI?",
    "Do you like pizza?",
    "Say something.",
]

def apply_chat_template(text):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": text}],
        tokenize=False,
        add_generation_prompt=True
    )

formatted_prompts = [apply_chat_template(p) for p in raw_prompts]
train_ds = Dataset.from_dict({"prompt": formatted_prompts * 10}) # 多复制几份

## 2.5 定义模型奖励函数 (Model-based Reward)

In [15]:
def reward_positive_sentiment(completions, **kwargs):
    """
    使用 BERT 模型来判断生成的回答够不够'Positive'
    """
    rewards = []
    
    # 提取纯文本回答
    texts = []
    for c in completions:
        # 这里还是用之前的逻辑提取 assistant 的回复
        if isinstance(c, list): 
            text = c[0]['content']
        else:
            text = str(c)
        # 截取掉 prompt 以外的部分 (根据具体 tokenizer 行为调整，这里简化处理)
        # 假设 completions 已经是纯文本或者我们主要看后半段
        # 为了防报错，取后 512 个字符扔给 BERT (BERT 有长度限制)
        texts.append(text[-512:]) 

    # 批量预测 (Batch inference)
    # pipe 返回格式: [{'label': 'POSITIVE', 'score': 0.99}, ...]
    results = sentiment_pipe(texts)

    for res in results:
        score = res['score']
        label = res['label']
        
        # 我们的目标是 POSITIVE
        if label == 'POSITIVE':
            # 如果是积极的，奖励就是它的置信度 (0.5 ~ 1.0)
            # 为了拉开差距，我们把分数放大: score * 2
            rewards.append(score * 2.0)
        else:
            # 如果是消极的 (NEGATIVE)，我们要惩罚
            # 同样放大惩罚: -score
            rewards.append(-score)
            
    return rewards


## 2.6 验证函数

In [16]:
@torch.no_grad()
def evaluate_model(header="Current Status"):
    print(f"\n--- {header} ---")
    model.eval()
    test_prompts = formatted_prompts[:3]
    for p_text in test_prompts:
        inputs = tokenizer(p_text, return_tensors="pt").to(device)
        out = model.generate(**inputs, max_new_tokens=50, do_sample=True, temperature=0.7)
        res = tokenizer.decode(out[0], skip_special_tokens=True).split("assistant\n")[-1].strip()
        
        # 用 Reward Model 测一下当前分数
        score_dict = sentiment_pipe(res[:512])[0]
        print(f"Output: {res}")
        print(f"Sentiment: {score_dict['label']} ({score_dict['score']:.4f})")
        print("-" * 10)
    model.train()

## 2.7 配置 RLOO (参数略微激进一点)

In [17]:
args = RLOOConfig(
    output_dir="./rloo_qwen_sentiment",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    learning_rate=2e-5,          # 稍微大一点的学习率
    max_steps=80,                
    logging_steps=10,
    max_prompt_length=128,
    max_completion_length=64,
    num_generations=4,
    temperature=1.0, # 增加随机性，让模型探索更多语气
)

trainer = RLOOTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    reward_funcs=reward_positive_sentiment, # 换成新的奖励函数
    processing_class=tokenizer,
)

# --- 运行 ---
print("Before Training:")
evaluate_model("Before")
trainer.train()
print("After Training:")
evaluate_model("After")

d:\anaconda\envs\api_model\lib\site-packages\trl\trainer\rloo_trainer.py:277: UserWarning: This trainer will soon be moved to trl.experimental and is a candidate for removal. If you rely on it and want it to remain, please share your comments here: https://github.com/huggingface/trl/issues/4223. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  warnings.warn(


Before Training:

--- Before ---
Output: I'm sorry for any inconvenience caused, but I am currently unable to provide information on real-time weather conditions as it goes beyond my capabilities as an AI language model. However, you can easily find up-to-date weather updates through various online sources such as
Sentiment: NEGATIVE (0.9990)
----------
Output: A cat is an animal belonging to the Felidae family. Cats are typically small to medium-sized mammals with retractable claws and long,柔软的爪子。They have sharp teeth and powerful jaws for hunting prey, which they use to catch mice
Sentiment: POSITIVE (0.9842)
----------


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Output: As the most advanced AI language model, I do not have a physical body or a personal life in the same way that humans do. However, my "day" is characterized by a series of tasks and operations designed to assist users with various inquiries and
Sentiment: NEGATIVE (0.9818)
----------


d:\anaconda\envs\api_model\lib\site-packages\torch\utils\checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss
10,0.000000
20,-0.000000
30,-0.000000
40,0.000000
50,0.000000
60,-0.000000
70,-0.000000
80,-0.000000


After Training:

--- After ---
Output: Weather forecast shows the weather of the sky. The weather forecast will tell you how it's going to rain, snow, or clear. Weather forecasts show the weather of the sky. The weather forecast tells us about the weather, weather, weather, weather
Sentiment: POSITIVE (0.9959)
----------
Output: Cats are the pets of humans. They are not only the pets of children, young people, seniors, seniors, seniors, seniors, seniors, seniors, seniors, seniors, seniors, seniors, seniors, seniors, seniors, seniors, seniors,
Sentiment: POSITIVE (0.8388)
----------
Output: The daily work of a user interface designer is to create the experience of a person's mind. The purpose of this role lies in creating a positive and positive environment for people. As such, we have been able to identify some of the important aspects that
Sentiment: POSITIVE (0.9998)
----------
